# Física BR · Taller: Circuitos de Corriente Directa
### Física BR — Física de Electricidad y Magnetismo
**Profesor:** Juan David Betancur Ríos · **Semestre:** 2026-2

---
*Física interactiva para ingeniería.* El análisis de circuitos de corriente directa (DC) es la
base del diseño eléctrico y electrónico. Cada punto parte de un problema real de ingeniería e
incluye esquema, sliders, botón **"Mostrar solución"** y cuestionario.

**Convenciones:** tres cifras significativas · coma decimal.

**Ecuaciones clave:**
- Serie:  R_eq = R₁ + R₂ + …    (misma corriente, los voltajes se suman)
- Paralelo:  1/R_eq = 1/R₁ + 1/R₂ + …    (mismo voltaje, las corrientes se suman)
- Fuente real:  V_terminal = fem − I·r    (r = resistencia interna)
- Ley de nodos (Kirchhoff):  Σ I_entra = Σ I_sale
- Circuito RC:  τ = R·C    (constante de tiempo de carga/descarga)

> **Uso:** `Entorno de ejecución → Ejecutar todo`. Competencias: [CL, UFT, UCM, PRC, ULC].

---
> © 2026 **Juan David Betancur Ríos**, docente de física para ingeniería. Material educativo de uso académico (Física BR). Todos los derechos reservados.
> *Elaborado con apoyo de herramientas de inteligencia artificial, bajo la supervisión y criterio pedagógico del autor.*

---
## Punto 1 — Resistencias en serie (Ing. Eléctrica/Electrónica)  ★★  [CL, UCM]
Tres resistencias  R₁ = 10,0 Ω,  R₂ = 20,0 Ω,  R₃ = 30,0 Ω  se conectan en serie a una fuente de
V = 120 V.

**Serie:**  R_eq = R₁ + R₂ + R₃    **Corriente:**  I = V/R_eq    **Caídas:**  Vᵢ = I·Rᵢ

> 🔧 **Aplicación en ingeniería:** en una conexión en serie la misma corriente pasa por todos los
> elementos y el voltaje se reparte. Es el principio de los divisores de tensión usados en sensores
> y circuitos de acondicionamiento de señal. Si un elemento se abre, toda la cadena deja de conducir.

> 🤖 **TIP-IA — Verificador crítico:**
> Antes de ver la solución, dile a un asistente: *"Calculé R_eq en serie y la corriente del circuito. Actúa como evaluador escéptico: pregúntame si en serie la resistencia total es mayor o menor que la más grande, y por qué la corriente es la misma en todas. No me des la respuesta."* Primero resuelves tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


def esquema():
    fig,ax=plt.subplots(figsize=(7.5,4))
    # circuito serie: batería y dos resistencias en línea
    ax.plot([1,1,7,7,1],[1,4,4,1,1],color='#2c3e50',lw=2)
    # batería
    ax.plot([0.8,1.2],[2.7,2.7],color='#c0392b',lw=4); ax.plot([0.85,1.15],[2.3,2.3],color='#2980b9',lw=2)
    ax.text(0.4,2.5,'V',fontsize=11,color='#2c3e50',fontweight='bold')
    # dos resistencias en la línea superior (zigzag)
    for cx,lbl,col in [(3,'R₁','#e67e22'),(5.5,'R₂','#8e44ad')]:
        xr=np.linspace(cx-0.6,cx+0.6,20); yr=4+0.15*np.sin(np.linspace(0,6*np.pi,20))
        ax.plot(xr,yr,color=col,lw=2.5); ax.text(cx,4.4,lbl,ha='center',fontsize=10,color=col,fontweight='bold')
    # corriente (la MISMA en todo el circuito)
    for x in [2,4,6]:
        ax.annotate('',xy=(x+0.4,1),xytext=(x,1),arrowprops=dict(arrowstyle='-|>',color='#27ae60',lw=2))
    ax.text(4,0.5,'MISMA corriente I en todo el circuito',ha='center',fontsize=9,color='#27ae60',fontweight='bold')
    ax.text(4,3.2,'El voltaje se REPARTE: V = V₁ + V₂',ha='center',fontsize=9,color='#2c3e50',fontweight='bold')
    ax.text(7.3,2.5,'R_total = R₁+R₂',fontsize=9,color='#c0392b',ha='left',rotation=90,va='center')
    ax.set_xlim(0,8.5); ax.set_ylim(0,5); ax.axis('off')
    ax.set_title('Resistencias en SERIE')
    plt.tight_layout(); plt.show()
esquema()

def serie(R1=10.0, R2=20.0, R3=30.0, V=120.0):
    Req=R1+R2+R3; I=V/Req
    plt.figure(figsize=(8,4))
    plt.bar(['R1','R2','R3','R_eq'],[R1,R2,R3,Req],color=['gray','gray','gray','navy'])
    plt.ylabel('Resistencia (Ω)'); plt.title(f'R_eq={Req:.0f} Ω  ·  I=V/R_eq={I:.2f} A'); plt.grid(alpha=.3,axis='y')
    plt.tight_layout(); plt.show()
    print(f"Caídas de voltaje: V1={I*R1:.1f} V, V2={I*R2:.1f} V, V3={I*R3:.1f} V (suman {I*Req:.0f} V)".replace(".",","))

interact(serie,
    R1=FloatSlider(value=10.0,min=5,max=50,step=5,description='R1 (Ω)'),
    R2=FloatSlider(value=20.0,min=5,max=50,step=5,description='R2 (Ω)'),
    R3=FloatSlider(value=30.0,min=5,max=50,step=5,description='R3 (Ω)'),
    V=FloatSlider(value=120.0,min=10,max=240,step=10,description='V (V)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)

    R1,R2,R3,V=10.0,20.0,30.0,120.0; Req=R1+R2+R3; I=V/Req
    _pc(f"R_eq = R1+R2+R3 = {Req:.3g} Ω")
    _pc(f"I = V/R_eq = 120/{Req:.0f} = {I:.3g} A (la misma en toda la serie)")
    _pc(f"Caídas: V1=I·R1={I*R1:.1f} V, V2={I*R2:.1f} V, V3={I*R3:.1f} V")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 1 ---")
quiz_numerico("Se conectan en serie tres resistencias de 10,0 Ω, 20,0 Ω y 30,0 Ω. Encuentre la resistencia equivalente del conjunto.",60,0.03,"Ω","R_eq=R1+R2+R3=60 Ω.", id_preg="Circ_p1_1", peso=2.0)
quiz_numerico("Si la resistencia equivalente en serie es de 60,0 Ω y la fuente entrega 120 V, encuentre la corriente que circula por el circuito.",2.0,0.04,"A","I=V/R_eq=2,00 A.", id_preg="Circ_p1_2", peso=2.0)
quiz_opcion_multiple("En un circuito en serie, ¿cómo es la corriente que pasa por cada resistencia?",
    ['Es distinta en cada una (según su R)', 'Es la misma en todas', 'Se reparte entre ellas', 'Es cero en la mayor'],1,
    "En serie hay un solo camino: la misma corriente pasa por todas.", id_preg="Circ_p1_3", peso=2.0)

# --- Estilo visual Física BR (apariencia global de los esquemas) ---
try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---

quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un estudiante dice que en un circuito en SERIE la resistencia de MAYOR valor recibe MENOS voltaje porque 'frena la corriente'. ¿Correcto o solo convincente?",
    ['Solo convincente: en serie I es común; con V = I·R, la mayor R recibe más voltaje',
     'Correcto: la resistencia mayor frena la corriente, y por eso le llega menos voltaje',
     'Correcto: por la resistencia mayor circula menos corriente que por las otras, y por eso sobre ella cae menos voltaje',
     'Solo convincente: en serie todas las resistencias reciben el mismo voltaje'],
    0,
    "En serie no hay 'más o menos corriente' por cada resistencia: es una sola corriente. El voltaje se reparte en proporción a R. Igual voltaje en todas ocurre en PARALELO.",
    id_preg='Circ_p1_concep_1', peso=2.0)


---
## Punto 2 — Resistencias en paralelo (Ing. Eléctrica/Instalaciones)  ★★★  [PRC, UCM]
Las mismas tres resistencias  R₁ = 10,0 Ω,  R₂ = 20,0 Ω,  R₃ = 30,0 Ω  se conectan ahora en
paralelo a  V = 120 V.

**Paralelo:**  1/R_eq = 1/R₁ + 1/R₂ + 1/R₃    **Ramas:**  Iᵢ = V/Rᵢ    **Total:**  I = ΣIᵢ

> 🔧 **Aplicación en ingeniería:** las instalaciones eléctricas domésticas e industriales conectan
> las cargas en paralelo, para que todas reciban el mismo voltaje y funcionen independientemente
> (si una falla, las demás siguen). Por eso la resistencia equivalente baja y la corriente total sube.

> 🤖 **TIP-IA — Abogado del diablo:**
> Dile a un asistente: *"Voy a sostener que conectar más resistencias en PARALELO aumenta la resistencia total, igual que en serie. Dame el mejor contraargumento."* (Pista: en paralelo abres más caminos para la corriente). La IA es espejo de tu razonamiento; primero decide tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


def esquema():
    fig,ax=plt.subplots(figsize=(7.5,4))
    # circuito paralelo
    ax.plot([1,1,6,6,1],[1,4,4,1,1],color='#2c3e50',lw=2)
    # batería
    ax.plot([0.8,1.2],[2.7,2.7],color='#c0392b',lw=4); ax.plot([0.85,1.15],[2.3,2.3],color='#2980b9',lw=2)
    ax.text(0.4,2.5,'V',fontsize=11,color='#2c3e50',fontweight='bold')
    # dos ramas en paralelo
    for x,lbl,col in [(3,'R₁','#e67e22'),(4.5,'R₂','#8e44ad')]:
        ax.plot([x,x],[4,1],color='#2c3e50',lw=1.5)
        yr=np.linspace(2,3,15); xr=x+0.15*np.sin(np.linspace(0,5*np.pi,15))
        ax.plot(xr,yr,color=col,lw=2.5); ax.text(x+0.3,2.5,lbl,fontsize=10,color=col,fontweight='bold')
    # corriente que se divide
    ax.annotate('',xy=(2.5,4),xytext=(2,4),arrowprops=dict(arrowstyle='-|>',color='#27ae60',lw=2))
    ax.text(2.2,4.3,'I',fontsize=10,color='#27ae60',fontweight='bold')
    ax.annotate('',xy=(3,3.6),xytext=(3,3.9),arrowprops=dict(arrowstyle='-|>',color='#27ae60',lw=1.5))
    ax.annotate('',xy=(4.5,3.6),xytext=(4.5,3.9),arrowprops=dict(arrowstyle='-|>',color='#27ae60',lw=1.5))
    ax.text(3.75,0.5,'La corriente se REPARTE: I = I₁ + I₂',ha='center',fontsize=9,color='#27ae60',fontweight='bold')
    ax.text(3.75,4.6,'MISMO voltaje en cada rama',ha='center',fontsize=9,color='#2c3e50',fontweight='bold')
    ax.text(6.3,2.5,'1/R_t = 1/R₁+1/R₂',fontsize=8,color='#c0392b',ha='left',rotation=90,va='center')
    ax.set_xlim(0,8); ax.set_ylim(0,5.2); ax.axis('off')
    ax.set_title('Resistencias en PARALELO')
    plt.tight_layout(); plt.show()
esquema()

def paralelo(R1=10.0, R2=20.0, R3=30.0, V=120.0):
    Req=1/(1/R1+1/R2+1/R3); I=V/Req
    plt.figure(figsize=(8,4))
    plt.bar(['R1','R2','R3','R_eq'],[R1,R2,R3,Req],color=['gray','gray','gray','orange'])
    plt.ylabel('Resistencia (Ω)'); plt.title(f'R_eq={Req:.2f} Ω  ·  I_total={I:.1f} A'); plt.grid(alpha=.3,axis='y')
    plt.tight_layout(); plt.show()
    print(f"Corrientes: I1={V/R1:.1f} A, I2={V/R2:.1f} A, I3={V/R3:.1f} A (suman {I:.1f} A)".replace(".",","))

interact(paralelo,
    R1=FloatSlider(value=10.0,min=5,max=50,step=5,description='R1 (Ω)'),
    R2=FloatSlider(value=20.0,min=5,max=50,step=5,description='R2 (Ω)'),
    R3=FloatSlider(value=30.0,min=5,max=50,step=5,description='R3 (Ω)'),
    V=FloatSlider(value=120.0,min=10,max=240,step=10,description='V (V)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)

    R1,R2,R3,V=10.0,20.0,30.0,120.0; Req=1/(1/R1+1/R2+1/R3); I=V/Req
    _pc(f"1/R_eq = 1/10+1/20+1/30 = 0,1+0,05+0,0333 = 0,1833")
    _pc(f"R_eq = {Req:.3g} Ω (menor que la resistencia más pequeña)")
    _pc(f"I_total = V/R_eq = 120/{Req:.2f} = {I:.3g} A")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 2 ---")
quiz_opcion_multiple("Tres resistencias se conectan primero en serie y luego en paralelo. ¿En qué caso la resistencia equivalente es MENOR?",['En serie (cada R aporta un camino)', 'En paralelo (cada rama añade un camino)', 'Es igual (mismas resistencias)', 'Depende del voltaje aplicado'],1,"En paralelo la equivalente es menor que la más pequeña, porque la corriente tiene varios caminos. En serie se suman. Es lo opuesto a los capacitores.", id_preg="Circ_p2_4", peso=3.0)
quiz_numerico("Si la resistencia equivalente en paralelo es de 5,45 Ω y la fuente entrega 120 V, encuentre la corriente total que entrega la fuente.",22,0.04,"A","I=V/R_eq=22,0 A.", id_preg="Circ_p2_5", peso=3.0)
quiz_opcion_multiple("Al conectar varias resistencias en paralelo, ¿cómo resulta la resistencia equivalente respecto a las individuales?",
    ['Mayor que todas (se suman)', 'Menor que la más pequeña', 'Igual a la suma de todas', 'El promedio de ellas'],1,
    "En paralelo R_eq es menor que la resistencia más pequeña.", id_preg="Circ_p2_6", peso=3.0)

quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un estudiante afirma que al añadir más resistencias en PARALELO la resistencia equivalente AUMENTA porque 'hay más resistencias'. ¿Correcto o solo convincente?",
    ['Solo convincente: cada rama abre otro camino; R_eq baja y queda por debajo de la rama menor',
     'Correcto: más resistencias siempre suman más oposición, estén como estén conectadas',
     'Correcto: en paralelo R_eq es la suma de las resistencias, igual que en serie',
     'Solo convincente: R_eq no cambia al añadir ramas, porque el voltaje sobre ellas es el mismo'],
    0,
    '1/R_eq = Σ1/Rᵢ: cada rama suma conductancia. Mismo voltaje con más ramas implica MÁS corriente total, es decir, menor R_eq.',
    id_preg='Circ_p2_concep_2', peso=2.0)


---
## Punto 3 — Fuente real con resistencia interna (Ing. Eléctrica/Electrónica)  ★★★  [PRC, ULC]
Una batería de  fem = 12,0 V  tiene resistencia interna  r = 0,500 Ω  y alimenta una carga
externa  R = 5,50 Ω.

**Corriente:**  I = fem/(R + r)    **Voltaje en terminales:**  V_term = fem − I·r

> 🔧 **Aplicación en ingeniería:** ninguna fuente real es ideal: su resistencia interna hace que el
> voltaje entregado baje al aumentar la corriente. Esto explica por qué una batería "cae" bajo carga
> pesada, y es clave al dimensionar baterías, UPS y fuentes para arranque de motores.

> 🤖 **TIP-IA — Traductor de representaciones:**
> Pega a un asistente: *"Explícame físicamente por qué el voltaje en los bornes de una batería BAJA cuando entrega más corriente, por su resistencia interna r. Dame la intuición, no solo la fórmula V=fem−I·r."* Primero intenta explicarlo tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


def esquema():
    fig,ax=plt.subplots(figsize=(7.5,4))
    # batería real = fem + resistencia interna r, dentro de una caja
    ax.add_patch(plt.Rectangle((1,1.5),2.5,2.5,fill=False,ec='#c0392b',lw=1.5,ls='--'))
    ax.text(2.25,4.2,'batería real',ha='center',fontsize=9,color='#c0392b',fontweight='bold')
    # fem
    ax.plot([1.5,1.9],[3.2,3.2],color='#2c3e50',lw=4); ax.plot([1.55,1.85],[2.8,2.8],color='#2c3e50',lw=2)
    ax.text(2.3,3,'fem ε',fontsize=9,color='#2c3e50')
    # resistencia interna
    yr=np.linspace(1.8,2.4,12); xr=2.5+0.1*np.sin(np.linspace(0,4*np.pi,12))
    ax.plot(xr,yr,color='#e67e22',lw=2); ax.text(2.8,2.1,'r interna',fontsize=8,color='#e67e22')
    # carga externa R
    ax.plot([3.5,6,6,3.5],[3.5,3.5,1.5,1.5],color='#2c3e50',lw=2)
    ax.plot([1,3.5],[3.5,3.5],color='#2c3e50',lw=2); ax.plot([1,3.5],[1.5,1.5],color='#2c3e50',lw=2)
    ax.plot([1,1],[1.5,3.5],color='#2c3e50',lw=2)
    yr2=np.linspace(2,3,12); ax.plot(6+0.12*np.sin(np.linspace(0,5*np.pi,12)),yr2,color='#8e44ad',lw=2.5)
    ax.text(6.3,2.5,'carga R',fontsize=9,color='#8e44ad')
    ax.text(3.5,0.6,'El voltaje entregado V = ε − I·r  (menor que la fem por la caída interna)',
            ha='center',fontsize=9,color='#2c3e50',fontweight='bold')
    ax.set_xlim(0,7.5); ax.set_ylim(0,4.6); ax.axis('off')
    ax.set_title('Fuente real: la resistencia interna reduce el voltaje')
    plt.tight_layout(); plt.show()
esquema()

def fuente(fem=12.0, r=0.5, R=5.5):
    I=fem/(R+r); Vt=fem-I*r
    Rs=np.linspace(0.5,20,100); Is=fem/(Rs+r); Vts=fem-Is*r
    plt.figure(figsize=(9,4.2)); plt.plot(Rs,Vts,lw=2,color='navy')
    plt.scatter([R],[Vt],color='red',s=60,zorder=5,label=f'R={R:.1f} Ω → I={I:.2f} A, V_term={Vt:.2f} V')
    plt.axhline(fem,color='gray',ls='--',alpha=.6,label=f'fem={fem:.0f} V (ideal)')
    plt.xlabel('resistencia de carga R (Ω)'); plt.ylabel('voltaje en terminales (V)')
    plt.title('El voltaje cae bajo carga (menor R ⟹ menor V_term)'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()

interact(fuente,
    fem=FloatSlider(value=12.0,min=1.5,max=24,step=1.5,description='fem (V)'),
    r=FloatSlider(value=0.5,min=0.1,max=2,step=0.1,description='r (Ω)'),
    R=FloatSlider(value=5.5,min=0.5,max=20,step=0.5,description='R (Ω)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)

    fem=12.0; r=0.5; R=5.5; I=fem/(R+r)
    _pc(f"Corriente: I = fem/(R+r) = 12,0/(5,50+0,500) = 12,0/6,00 = {I:.3g} A")
    _pc(f"Voltaje en terminales: V_term = fem − I·r = 12,0 − {I:.1f}·0,500 = {fem-I*r:.3g} V")
    _pc(f"Verificación: V_term = I·R = {I:.1f}·5,50 = {I*R:.3g} V ✓")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 3 ---")
quiz_numerico("Una batería de fem 12,0 V y resistencia interna 0,500 Ω alimenta una carga de 5,50 Ω. Encuentre la corriente que circula por el circuito.",2.0,0.04,"A","I=fem/(R+r)=2,00 A.", id_preg="Circ_p3_7", peso=3.0)
quiz_opcion_multiple("Una batería real tiene resistencia interna. ¿Por qué el voltaje en sus bornes BAJA cuando entrega mucha corriente?",['Porque la batería se agota en cuanto entrega mucha corriente', 'Parte de la fem cae en la resistencia interna (V = fem − I·r)', 'Porque la fem de la batería cambia cuando aumenta la corriente', 'No baja nunca: una batería ideal y una real se comportan igual'],1,"El voltaje útil es V=fem−I·r. A mayor corriente, mayor caída I·r dentro de la batería, y menor voltaje en bornes.", id_preg="Circ_p3_8", peso=3.0)
quiz_opcion_multiple("En una fuente real con resistencia interna, si aumenta la corriente que entrega, ¿qué ocurre con el voltaje en sus terminales?",
    ['Sube (más corriente, más voltaje)', 'Baja (V = fem − I·r)', 'No cambia (la fem es fija)', 'Se anula (cortocircuito)'],1,
    "Mayor I ⟹ mayor caída I·r ⟹ menor voltaje entregado.", id_preg="Circ_p3_9", peso=3.0)

quiz_opcion_multiple('DEFIENDE TU CONCLUSIÓN: un estudiante dice que una batería de fem 12 V entrega MENOS de 12 V en sus bornes cuando alimenta una carga. ¿Correcto o solo convincente?',
    ['Correcto: V_bornes = fem − I·r; con corriente, la resistencia interna hace caer el voltaje disponible',
     'Correcto: entrega menos porque la fem de la batería se va gastando a medida que circula la corriente por la carga',
     'Solo convincente: la fem es el voltaje en bornes, así que siempre entrega 12 V',
     'Solo convincente: entrega más de 12 V cuando la carga le exige mucha corriente'],
    0,
    "Esta vez es CORRECTO. La fem no 'se gasta' con la corriente: es la r interna la que produce la caída I·r. Por eso una batería vieja (r alta) 'no da fuerza' aunque marque 12 V en vacío.",
    id_preg='Circ_p3_concep_3', peso=2.0)


---
## Punto 4 — Ley de nodos de Kirchhoff (Ing. Eléctrica/Sistemas)  ★★  [CL, PRC]
En un nodo de una red eléctrica entra una corriente  I₁ = 8,00 A  y por una rama sale
I₂ = 3,00 A. Se busca la corriente  I₃  por la otra rama.

**Ley de nodos:**  Σ I_entra = Σ I_sale    →    I₁ = I₂ + I₃

> 🔧 **Aplicación en ingeniería:** la ley de nodos expresa la conservación de la carga: lo que entra
> a un nodo sale. Es la base para analizar redes eléctricas complejas (tableros de distribución,
> circuitos con múltiples ramas) y verificar que las corrientes estén balanceadas.

> 🤖 **TIP-IA — Verificador crítico:**
> Dile a un asistente: *"Usé la ley de nodos de Kirchhoff para hallar I₃. Actúa como evaluador escéptico: pregúntame de dónde sale la ley de nodos (qué principio físico la sostiene) y por qué la suma de corrientes que entran es igual a la que sale. No me des la respuesta."*


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


def esquema():
    fig,ax=plt.subplots(figsize=(7,4))
    # nodo central
    nodo=(3.5,2)
    ax.scatter([nodo[0]],[nodo[1]],s=120,color='#2c3e50',zorder=6)
    ax.text(nodo[0],nodo[1]+0.3,'nodo',fontsize=9,color='#2c3e50',fontweight='bold')
    # corriente que entra
    ax.annotate('',xy=(nodo[0]-0.3,nodo[1]),xytext=(1,nodo[1]),arrowprops=dict(arrowstyle='-|>',color='#27ae60',lw=3))
    ax.text(1.5,2.3,'I₁ (entra)',fontsize=10,color='#27ae60',fontweight='bold')
    # dos corrientes que salen
    ax.annotate('',xy=(5.5,3.3),xytext=(nodo[0]+0.2,nodo[1]+0.2),arrowprops=dict(arrowstyle='-|>',color='#e74c3c',lw=2.5))
    ax.text(5.3,3.5,'I₂ (sale)',fontsize=10,color='#e74c3c',fontweight='bold')
    ax.annotate('',xy=(5.5,0.7),xytext=(nodo[0]+0.2,nodo[1]-0.2),arrowprops=dict(arrowstyle='-|>',color='#e74c3c',lw=2.5))
    ax.text(5.3,0.5,'I₃ (sale)',fontsize=10,color='#e74c3c',fontweight='bold')
    ax.text(3.5,0,'Lo que ENTRA = lo que SALE:  I₁ = I₂ + I₃  (conservación de carga)',
            ha='center',fontsize=9,color='#2c3e50',fontweight='bold')
    ax.set_xlim(0.5,6.5); ax.set_ylim(-0.4,4); ax.axis('off')
    ax.set_title('Ley de nodos de Kirchhoff')
    plt.tight_layout(); plt.show()
esquema()

def nodo(I1=8.0, I2=3.0):
    I3=I1-I2
    plt.figure(figsize=(7,4))
    plt.bar(['I1 (entra)','I2 (sale)','I3 (sale)'],[I1,I2,I3],color=['blue','red','green'])
    plt.ylabel('Corriente (A)'); plt.title(f'I3 = I1 − I2 = {I3:.2f} A'); plt.grid(alpha=.3,axis='y')
    plt.tight_layout(); plt.show()

interact(nodo,
    I1=FloatSlider(value=8.0,min=1,max=15,step=0.5,description='I1 (A)'),
    I2=FloatSlider(value=3.0,min=0.5,max=10,step=0.5,description='I2 (A)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)

    I1=8.0; I2=3.0
    _pc(f"Ley de nodos: I1 = I2 + I3")
    _pc(f"I3 = I1 − I2 = 8,00 − 3,00 = {I1-I2:.3g} A")
    _pc("La corriente que entra al nodo es igual a la que sale (conservación de carga).")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 4 ---")
quiz_numerico("A un nodo de una red entra una corriente de 8,00 A y por una rama sale una corriente de 3,00 A. Encuentre la corriente que circula por la otra rama.",5.0,0.04,"A","I3=I1−I2=5,00 A.", id_preg="Circ_p4_10", peso=2.0)
quiz_opcion_multiple("La ley de nodos de Kirchhoff (lo que entra a un nodo es igual a lo que sale) es una expresión de la conservación de:",
    ['La energía de las cargas', 'La carga eléctrica', 'El voltaje en cada rama', 'La resistencia total'],1,
    "Lo que entra al nodo sale: es conservación de la carga.", id_preg="Circ_p4_11", peso=2.0)

quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un estudiante afirma que en un nodo puede 'acumularse carga', así que la corriente que entra no tiene que ser igual a la que sale. ¿Correcto o solo convincente?",
    ['Solo convincente: la ley de nodos es conservación de carga; un nodo no acumula carga',
     'Correcto: los nodos se comportan como pequeños capacitores que almacenan durante un tiempo la carga sobrante del circuito',
     'Correcto: donde se juntan varias corrientes, la carga se acumula hasta encontrar salida',
     'Solo convincente: la corriente que sale siempre es mayor, porque el nodo suma las ramas'],
    0,
    'Si un nodo acumulara carga, su potencial cambiaría sin parar: no habría régimen estacionario. ΣI_entra = ΣI_sale. La suma de ramas se reparte; no crea corriente extra.',
    id_preg='Circ_p4_concep_4', peso=2.0)


---
## Punto 5 — Constante de tiempo de un circuito RC (Ing. Electrónica/Control)  ★★★  [PRC, ULC]
Un capacitor  C = 100 µF  se carga a través de una resistencia  R = 1000 Ω. Se estudia la rapidez
de carga mediante la constante de tiempo.

**Constante de tiempo:**  τ = R·C    (en t = τ el capacitor alcanza 63,2% de su carga final;
en t = 5τ, ~99%)

> 🔧 **Aplicación en ingeniería:** los circuitos RC fijan tiempos en temporizadores, filtros,
> antirrebote de pulsadores y arranques suaves. Elegir R y C define cuánto tarda una señal en
> estabilizarse: es diseño fundamental en electrónica y sistemas de control.

> 🤖 **TIP-IA — Generador de problemas:**
> Pídele: *"Créame una variante de este circuito RC con otra R y otro C, y pídeme calcular la constante de tiempo τ=RC y el tiempo para cargar al 63%. No me des la solución hasta que yo la intente."* Primero resuelves tú, después verificas.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


def esquema():
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(9,4))
    # carga del capacitor (sube exponencial)
    t=np.linspace(0,5,100); Vc=1-np.exp(-t)
    ax1.plot(t,Vc,color='#2980b9',lw=2.5)
    ax1.axhline(1,color='#27ae60',ls='--',lw=1); ax1.text(4,1.03,'V máximo',fontsize=8,color='#27ae60')
    ax1.axvline(1,color='#7f8c8d',ls=':',lw=1)
    ax1.scatter([1],[1-np.exp(-1)],color='#f39c12',s=60,zorder=5)
    ax1.text(1.1,0.5,'τ=RC\n(63%)',fontsize=8,color='#b9770e')
    ax1.set_xlabel('tiempo'); ax1.set_ylabel('voltaje del capacitor')
    ax1.set_title('Carga: sube hasta el máximo',fontsize=10); ax1.grid(alpha=0.2,linestyle='--')
    # descarga (baja exponencial)
    Vd=np.exp(-t)
    ax2.plot(t,Vd,color='#c0392b',lw=2.5)
    ax2.axvline(1,color='#7f8c8d',ls=':',lw=1)
    ax2.scatter([1],[np.exp(-1)],color='#f39c12',s=60,zorder=5)
    ax2.text(1.1,0.5,'τ=RC\n(37%)',fontsize=8,color='#b9770e')
    ax2.set_xlabel('tiempo'); ax2.set_ylabel('voltaje del capacitor')
    ax2.set_title('Descarga: baja hasta cero',fontsize=10); ax2.grid(alpha=0.2,linestyle='--')
    plt.suptitle('Circuito RC: la constante de tiempo τ=R·C marca el ritmo',fontsize=11,fontweight='bold',color='#2c3e50')
    plt.tight_layout(); plt.show()
esquema()

def rc(R=1000.0, C_uF=100.0, V=9.0):
    C=C_uF*1e-6; tau=R*C
    t=np.linspace(0,5*tau,200); q=V*C*(1-np.exp(-t/tau))
    plt.figure(figsize=(9,4.2)); plt.plot(t,q*1e6,lw=2,color='teal')
    plt.axvline(tau,color='red',ls='--',alpha=.6,label=f'τ=RC={tau:.3f} s (63,2%)')
    plt.axvline(5*tau,color='gray',ls='--',alpha=.5,label=f'5τ={5*tau:.2f} s (~99%)')
    plt.xlabel('tiempo t (s)'); plt.ylabel('carga q (µC)')
    plt.title(f'Carga del capacitor  ·  τ={tau:.3f} s'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()

interact(rc,
    R=FloatSlider(value=1000.0,min=100,max=5000,step=100,description='R (Ω)'),
    C_uF=FloatSlider(value=100.0,min=10,max=500,step=10,description='C (µF)'),
    V=FloatSlider(value=9.0,min=1.5,max=24,step=1.5,description='V (V)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)

    R=1000.0; C=100e-6; tau=R*C
    _pc(f"Constante de tiempo: τ = R·C = 1000·100e-6 = {tau:.3g} s")
    _pc(f"En t=τ={tau:.2f} s el capacitor alcanza el 63,2% de su carga final.")
    _pc(f"En t=5τ={5*tau:.2f} s alcanza ~99% (se considera 'cargado').")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 5 ---")
quiz_numerico("Un capacitor de 100 µF se carga a través de una resistencia de 1000 Ω. Encuentre la constante de tiempo del circuito RC.",0.1,0.05,"s","τ=RC=0,100 s.", id_preg="Circ_p5_12", peso=3.0)
quiz_opcion_multiple("En un circuito RC, cuando ha transcurrido un tiempo igual a una constante de tiempo (t = τ), ¿a qué porcentaje de su carga final ha llegado el capacitor?",
    ["100%","63,2%","50%","0%"],1,"En t=τ el capacitor alcanza 63,2% de su carga final.", id_preg="Circ_p5_13", peso=3.0)
quiz_opcion_multiple("En un circuito RC, la constante de tiempo es τ=RC. ¿Qué representa físicamente un τ grande?",['Que el capacitor se carga muy rápido, porque τ mide su velocidad', 'Que el capacitor se carga lentamente (más R o más C)', 'Que el capacitor no llega a cargarse nunca, sin importar el tiempo', 'Que el voltaje final del capacitor será mayor que la fem'],1,"τ=RC marca el ritmo de carga: un τ grande significa carga lenta. Tras un τ el capacitor alcanza ~63%; tras 5τ se considera cargado.", id_preg="Circ_p5_14", peso=3.0)

quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un estudiante dice que aumentar la resistencia R de un circuito RC hace que el capacitor se cargue MÁS RÁPIDO porque 'llega más energía'. ¿Correcto o solo convincente?",
    ['Solo convincente: τ = R·C; una R mayor limita la corriente de carga y el capacitor tarda más',
     'Correcto: con más R se disipa más energía en el circuito, y esa energía extra llena antes el capacitor',
     'Correcto: la constante de tiempo τ = C/R disminuye cuando R aumenta',
     'Solo convincente: R no influye en el tiempo de carga, solo en el voltaje final'],
    0,
    'La energía disipada en R se pierde como calor, no llega al capacitor. El voltaje final es la fem sin importar R; lo que R cambia es el TIEMPO (τ = RC).',
    id_preg='Circ_p5_concep_5', peso=2.0)


---
## ✅ Fin del taller de Circuitos DC (Física BR)
Cada punto nació de un problema real: divisores de tensión, instalaciones en paralelo, baterías
bajo carga, redes con nodos y temporizadores RC. Experimenta con los sliders y **predice** antes
de pulsar "Mostrar solución".
*Física interactiva para ingeniería — 2026-2.*

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


# Calificación formativa del taller
calificacion_final(total_preguntas=19)
